In [1]:
"""
S01 MPO training — integrate notebooks 01–07 mission context.

Scenario:
  - Same mission profile as notebook 07 baseline overflight (50-target meridian grid,
    seeded clouds over the corridor, dual camera, attitude safety on, image quality)
  - Controller features selected via ControllerFeatureConfig (see cell below)
  - MPO warmup (random + baseline) → 10 train episodes → eval
  - Preflight: inline feature checks + full ML training pytest suite

Verification: s01_utils/training_workflow.py
Artifacts: autonomous_control/models/nb-s01-08-<timestamp>/
Export: eval_best.mp4 in run directory
"""

'\nS01 MPO training — integrate notebooks 01–07 mission context.\n\nScenario:\n  - Same mission profile as notebook 07 baseline overflight (50-target meridian grid,\n    seeded clouds over the corridor, dual camera, attitude safety on, image quality)\n  - Controller features selected via ControllerFeatureConfig (see cell below)\n  - MPO warmup (random + baseline) → 10 train episodes → eval\n  - Preflight: inline feature checks + full ML training pytest suite\n\nVerification: s01_utils/training_workflow.py\nArtifacts: autonomous_control/models/nb-s01-08-<timestamp>/\nExport: eval_best.mp4 in run directory\n'

In [2]:
import os
import sys
from pathlib import Path

notebook_dir = Path.cwd()
backend_root = notebook_dir
for _ in range(6):
    if (backend_root / "simulation").is_dir():
        break
    backend_root = backend_root.parent
os.chdir(backend_root)
sys.path.insert(0, str(backend_root))
_s01_dir = backend_root / "notebooks" / "s01"
sys.path.insert(0, str(_s01_dir))
print(f"backend_root={backend_root}")

backend_root=d:\code\sem-proj-asc\backend


In [3]:
import importlib

import s01_utils.training_workflow as tw

importlib.reload(tw)

if tw.run_s01_training_preflight_gate():
    print("Training preflight gate passed (inline checks + pytest suite).")

c:\Users\cedri\miniconda3\envs\auto-sat\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Using device: cuda
Using device: cuda
Training preflight gate passed (inline checks + pytest suite).


In [4]:
from autonomous_control.feature_selection import ControllerFeatureConfig

# ── EDIT HERE: controller observation features ───────────────────────────────
# Keys must exist on SimulationTimestepState (see autonomous_control/feature_selection.py).
FEATURE_CONFIG = ControllerFeatureConfig(
    attitude_keys=(
        "body_z_angle_rad",
        "omega_sat_rad_s",
    ),
    orbit_keys=(
        "theta_orbit_rad",
    ),
    vision_keys=(
        "camera_observation_line_codes",
        "secondary_camera_observation_line_codes",
    ),
)

WORKFLOW_CONFIG = tw.TrainingWorkflowConfig(
    seed=7,
    train_episodes=100,
    feature_config=FEATURE_CONFIG,
)

# Resolve secondary camera bin count for layout tables (same mission profile as training).
_secondary_bins = int(
    tw.build_s01_training_mission_setup(seed=WORKFLOW_CONFIG.seed)
    .resolve(require_camera=True)
    .secondary_camera_observation_line_n_bins
)
tw.display_feature_tables(FEATURE_CONFIG, secondary_camera_bins=_secondary_bins)

### Controller feature selection (`ControllerFeatureConfig`)

Edit `FEATURE_CONFIG` in the notebook cell below, or change `S01_TRAINING_FEATURE_CONFIG` in `s01_utils/training_workflow.py`.

,group,timestep_key,state_dims,unit,encoder_path,source_module
0,attitude,body_z_angle_rad,1,rad,scalar → MLP branch,autonomous_control/feature_selection.py
1,attitude,omega_sat_rad_s,1,rad/s,scalar → MLP branch,autonomous_control/feature_selection.py
2,orbit,theta_orbit_rad,1,rad,scalar → MLP branch,autonomous_control/feature_selection.py
3,vision,camera_observation_line_codes,100,obs code / bin,int8 codes → embed → 1D-CNN,autonomous_control/feature_selection.py
4,vision,secondary_camera_observation_line_codes,200,obs code / bin,int8 codes → embed → 1D-CNN,autonomous_control/feature_selection.py


### Controller encoder routing (scalar **3**, vision streams **2**)

,stage,inputs,input_shape,module
0,scalar branch,"body_z_angle_rad, omega_sat_rad_s, theta_orbit...","(3,) float32",MLP → 90-D
1,vision branch (primary (nadir)),camera_observation_line_codes,"(100,) int8",ObservationLineCNNEncoder → 32-D
2,vision branch (secondary (forward)),secondary_camera_observation_line_codes,"(200,) int8",ObservationLineCNNEncoder → 32-D
3,vision fusion,concat CNN embeddings,"(64,)",MLP → 90-D
4,policy / Q trunk,"concat(scalar, vision)","(180,)",Actor head / Critic head


### Observation code legend (vision line bins)

,code,label
0,-99,not_computed
1,0,space
2,1,earth
3,2,cloud
4,3,target


In [5]:
setup = tw.build_training_workflow_setup(WORKFLOW_CONFIG)
tw.print_training_setup_summary(setup)
tw.display_feature_snapshot_tables(setup)

Using device: cuda
S01 MPO training setup (notebook 07 baseline overflight profile)
  run_dir:           D:\code\sem-proj-asc\backend\autonomous_control\models\nb-s01-08-2026-06-24_23-16-21
  seed:              7
  altitude:          528.8 km
  targets:           50
  clouds:            25 (seeded over target corridor)
  orbit window:      -32.7° .. 37.1°
  episode steps:     2904
  attitude safety:   on (training_episode_simulation_config)
  scalar dim:        3
  vision streams:    [('camera_observation_line_codes', 100), ('secondary_camera_observation_line_codes', 200)]
  encoder trunk:     180-D
  secondary bins:    200
  feature keys:      ['body_z_angle_rad', 'omega_sat_rad_s', 'theta_orbit_rad', 'camera_observation_line_codes', 'secondary_camera_observation_line_codes']
  warmup episodes:   4
  train episodes:    100
  eval episodes:     2
  MPO batch_size:    256
  MPO gamma:         0.99
  MPO LRs q/pi/eta:  0.00045/0.00015/0.001
  target_kl mu/sigma: 0.1/0.0001
  reward flags

d:\code\sem-proj-asc\backend\simulation\stepper.py:171: UserWarning: controller_update_interval (1 s) is not an integer multiple of simulation_timestep (0.4 s); using nearest multiple: 0.8 s (2 sim steps).
  self._controller_interval_steps, self._effective_controller_interval_s = resolve_controller_interval_steps(


### Scalar features at episode start (MLP branch)

,scalar_index,group,timestep_key,value,unit
0,0,attitude,body_z_angle_rad,4.111343,rad
1,1,attitude,omega_sat_rad_s,0.000000,rad/s
2,2,orbit,theta_orbit_rad,0.969750,rad


### Vision line features at episode start (CNN branches)

,camera,timestep_key,n_bins,preview,dominant_code,target_bins
0,primary (nadir),camera_observation_line_codes,100,"[1, 1, 1, 1, 1, 1, 1, 1, …]",1 (earth),0
1,secondary (forward),secondary_camera_observation_line_codes,200,"[1, 1, 1, 1, 1, 1, 1, 1, …]",1 (earth),0


Structured `ControllerObservation`: scalars **3**, vision **camera_observation_line_codes 100 bins, secondary_camera_observation_line_codes 200 bins** → encoder trunk **180**-D.

In [6]:
result = tw.run_training_workflow(setup, show_progress=True)

Warmup:   0%|          | 0/4 [00:00<?, ?ep/s]

╭──────────────────────────────────────────────── Simulation info ────────────────────────────────────────────────╮
│  Parameter                                     Value                                                            │
│  episode duration                              1160.93 s                                                        │
│  simulation timestep                           0.399906 s                                                       │
│  integration steps                             2903 (+1 state samples)                                          │
│  orbit altitude                                528.76 km                                                        │
│  orbit period                                  5703.8 s                                                         │
│  theta center offset                           90.00 deg                                                        │
│  episode theta start (rel. center)             -32.693 deg                                                      │
│  episode theta end (rel. center)               37.091 deg                                                       │
│  sat motion span scale                         1.050                                                            │
│  sat z offset                                  0.00 deg                                                         │
│  target areas                                  50                                                               │
│  target phi stripe                             79.93 deg .. 104.46 deg                                          │
│  cloud patches                                 25                                                               │
│  render mode                                   headless                                                         │
│  torque command source                         external                                                         │
│  torque policy                                 MPOAgent:warmup                                                  │
│  attitude controller                           enabled                                                          │
│  control stack (display)                       MPOAgent:warmup · attitude_controller                            │
│  controller seed                               -                                                                │
│  controller update (configured)                1 s                                                              │
│  controller update (effective)                 0.8 s (2 steps)                                                  │
│  reaction-wheel torque max                     0.1000 N*m                                                       │
│  attitude safety cutoff                        |omega_sat| > 3.00 deg/s -> block opposing torque                │
│  camera kernel backend                         accelerated                                                      │
│  reward shaping                                distance_reward, outer_gate                                      │
│  episode runner mode                           warmup                                                           │
│  agent                                         MPOAgent                                                         │
│  Camera 1 (primary) - role                     shapes reward (observation line + strip)                         │
│  Camera 1 (primary) - tilt off nadir           0.00 deg                                                         │
│  Camera 1 (primary) - FOV (cross x along)      1.61 deg x 1.20 deg                                              │
│  Camera 1 (primary) - resolution               9344 x 7000 px                                                   │
│  Camera 1 (primary) - pixel pitch              3.20 um                                                          │
│  Camera 1 (primary) - focal length             1067.0 

╭──────────────────────────────────────────────── Live stats ────────────────────────────────────────────────╮
│  Parameter       Value                                                                                     │
│  episode         warmup ep 1 / 4                                                                           │
│  step            400 / 2903 (13.8%)                                                                        │
│  sim time        160.0 s                                                                                   │
│  step reward     -100.0000                                                                                 │
│  episode return  -40000.00                                                                                 │
│  body z angle    -2.1087 rad                                                                               │
│  omega sat       -0.0039 rad/s                                                                             │
│  image smear     0.612 px                                                                                  │
│  image quality   0.2350                                                                                    │
│  buffer          399                                                                                       │
│  agent steps     399                                                                                       │
╰────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Live stats ────────────────────────────────────────────────╮
│  Parameter       Value                                                                                     │
│  episode         warmup ep 1 / 4                                                                           │
│  step            800 / 2903 (27.6%)                                                                        │
│  sim time        319.9 s                                                                                   │
│  step reward     -100.0000                                                                                 │
│  episode return  -80000.00                                                                                 │
│  body z angle    -1.8231 rad                                                                               │
│  omega sat       -0.0028 rad/s                                                                             │
│  image smear     0.575 px                                                                                  │
│  image quality   0.2475                                                                                    │
│  buffer          799                                                                                       │
│  agent steps     799                                                                                       │
╰────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Live stats ────────────────────────────────────────────────╮
│  Parameter       Value                                                                                     │
│  episode         warmup ep 1 / 4                                                                           │
│  step            1200 / 2903 (41.3%)                                                                       │
│  sim time        479.9 s                                                                                   │
│  step reward     -100.0000                                                                                 │
│  episode return  -109400.00                                                                                │
│  body z angle    -1.6437 rad                                                                               │
│  omega sat       0.0014 rad/s                                                                              │
│  image smear     0.432 px                                                                                  │
│  image quality   0.3044                                                                                    │
│  buffer          1199                                                                                      │
│  agent steps     1199                                                                                      │
╰────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

train ep 72:   1%|          | 19/2903 [00:02<05:05,  9.45step/s, reward=-100.000, total=-1900.0]

d:\code\sem-proj-asc\backend\notebooks\s01\s01_utils\training_workflow.py:767: UserWarning: [run_serial] WARNING: episode ended early — capture budget exhausted at step 19/2903 (mode=train)
  return setup.runner.run_serial(
Train:  72%|███████▏  | 72/100 [02:30<00:57,  2.06s/ep, kl=334594.0526, reward=-1900.0]


[run_serial] WARNING: episode ended early — capture budget exhausted at step 19/2903 (mode=train)
[run_serial] end mode=train steps=19 total_reward=-1900.000000 avg_reward=-100.000000
train ep 73:   1%|          | 15/2903 [00:01<05:05,  9.46step/s]

Train:  72%|███████▏  | 72/100 [02:32<00:59,  2.11s/ep, kl=334594.0526, reward=-1900.0]

KeyboardInterrupt: 

In [ ]:
tw.print_training_kpis(result)

In [ ]:
from utils.notebook.video import init_video_cell, play_saved_video

init_video_cell()

from utils.notebook.video import init_video_cell, play_saved_video

init_video_cell()
tw.display_training_artifacts(result)
video_path = result.artifact_paths["eval_best_video"]
if video_path.exists():
    play_saved_video(video_path)